Data Cropping

In [ ]:
import os
from PIL import Image

img_dir = "C:/Users/Mardyson Justin/Thesis/VisDrone2019-DET-train/images"
ann_dir = "C:/Users/Mardyson Justin/Thesis/VisDrone2019-DET-train/annotations"

out_dir = "C:/Users/Mardyson Justin/Thesis/VisDrone_SR/HRDelete"

os.makedirs(out_dir, exist_ok=True)

count = 0

for file in os.listdir(img_dir):
    if not file.endswith(".jpg"):
        continue

    img = Image.open(os.path.join(img_dir, file)).convert("RGB")
    W, H = img.size

    ann_path = os.path.join(ann_dir, file.replace(".jpg", ".txt"))
    if not os.path.exists(ann_path):
        continue

    with open(ann_path) as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()

        if not line:
            continue

        parts = line.split(",")

        # ensures correct VisDrone format length
        if len(parts) < 8:
            continue

        try:
            x, y, w, h = map(int, parts[:4])
            cls = int(parts[5])
        except ValueError:
            continue

        # skip ignored regions (class 0)
        if cls == 0:
            continue

        # padding around object
        pad = max(8, min(w, h) // 2)

        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(W, x + w + pad)
        y2 = min(H, y + h + pad)

        # ensures valid crop
        if x2 <= x1 or y2 <= y1:
            continue

        crop = img.crop((x1, y1, x2, y2))

        save_path = os.path.join(out_dir, f"{count:06d}.png")
        crop.save(save_path)

        count += 1

print("✅ HR-only SRNO dataset created:", count)


✅ HR-only SRNO dataset created: 344737


SRNO Training

In [ ]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

class SRImplicitDataset(Dataset):
    def __init__(self, img_dir, max_images=100):
        self.files = sorted([
            os.path.join(img_dir, f)
            for f in os.listdir(img_dir)
            if f.endswith(".png") or f.endswith(".jpg")
        ])[:max_images]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        hr = TF.to_tensor(img)

        h, w = hr.shape[1:]

        # HARD CLAMP HR (NO SKIP)
        h = max(2, h)
        w = max(2, w)
        hr = TF.resize(hr, (h, w), antialias=True)

        scale = random.uniform(1.5, 4.0)

        # SAFE LR (MIN = 2)
        lr_h = max(2, int(h / scale))
        lr_w = max(2, int(w / scale))

        lr = TF.resize(hr, (lr_h, lr_w), antialias=True)

        return lr, hr

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SRNOInspired(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, hidden, 3, padding=1),
            nn.ReLU()
        )

        self.mlp = nn.Sequential(
            nn.Linear(hidden + 2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 3)
        )

    def forward(self, lr, out_h, out_w):

        feat = self.encoder(lr)  # (B,C,H,W)
        B, C, H, W = feat.shape

        # -----------------------------
        # NORMALIZED COORDINATES [-1,1]
        # -----------------------------
        y = torch.linspace(-1, 1, out_h, device=lr.device)
        x = torch.linspace(-1, 1, out_w, device=lr.device)
        yy, xx = torch.meshgrid(y, x, indexing="ij")

        grid = torch.stack((xx, yy), dim=-1)  # (H,W,2)
        grid = grid.unsqueeze(0).repeat(B,1,1,1)

        # -----------------------------
        # SAMPLE FEATURES
        # -----------------------------
        sampled_feat = F.grid_sample(
            feat,
            grid,
            mode="bilinear",
            align_corners=False
        )  # (B,C,H_out,W_out)

        sampled_feat = sampled_feat.permute(0,2,3,1).reshape(-1, C)

        coords = grid.reshape(-1,2)

        # -----------------------------
        # MLP
        # -----------------------------
        inp = torch.cat([sampled_feat, coords], dim=1)
        out = self.mlp(inp)

        return out.view(B, out_h, out_w, 3).permute(0,3,1,2)

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Current device:", torch.cuda.current_device() if torch.cuda.is_available() else "CPU")
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


CUDA available: True
Current device: 0
Device name: NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = SRImplicitDataset(
    "C:/Users/Mardyson Justin/Thesis/VisDrone_SR/HRDelete", #dataset of cropped images
    max_images=120000
)
loader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=0, pin_memory=True)

model = SRNOInspired().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.L1Loss()

epochs = 30
save_dir = "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints5/120k_30e"
os.makedirs(save_dir, exist_ok=True)

# Di pede batch size gawa different sizes ang cropped images
accumulation_steps = 4
scaler = torch.amp.GradScaler("cuda")

best_loss = float("inf")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for i, (lr, hr) in enumerate(tqdm(loader)):
        lr, hr = lr.to(device), hr.to(device)
        _, _, H, W = hr.shape

        with torch.amp.autocast("cuda"):
            pred = model(lr, H, W)
            loss = loss_fn(pred, hr) / accumulation_steps

        scaler.scale(loss).backward()
        total_loss += loss.item() * accumulation_steps

        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

    if (i + 1) % accumulation_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.6f}")

    # -------- Save latest checkpoint --------
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "loss": avg_loss
    }

    torch.save(checkpoint, os.path.join(save_dir, "latest_checkpoint.pth"))

    # -------- Save per-epoch checkpoint --------
    torch.save(checkpoint,
               os.path.join(save_dir, f"checkpoint_epoch_{epoch+1}.pth"))

    # -------- Save best model --------
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(),
                   os.path.join(save_dir, "best_model.pth"))
        print("🔥 Best model updated!")

print("✅ Training complete.")


100%|██████████| 120000/120000 [23:47<00:00, 84.06it/s] 


Epoch 1 | Loss: 0.039773
🔥 Best model updated!


100%|██████████| 120000/120000 [22:55<00:00, 87.22it/s] 


Epoch 2 | Loss: 0.036804
🔥 Best model updated!


100%|██████████| 120000/120000 [23:33<00:00, 84.89it/s] 


...

Epoch 28 | Loss: 0.034329
🔥 Best model updated!


100%|██████████| 120000/120000 [22:12<00:00, 90.05it/s] 


Epoch 29 | Loss: 0.034309
🔥 Best model updated!


100%|██████████| 120000/120000 [26:45<00:00, 74.72it/s] 


Epoch 30 | Loss: 0.034324
✅ Training complete.


Using SRNO

In [ ]:
import torch
from PIL import Image
import torchvision.transforms.functional as TF
import os
import glob

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1️ Recreate model
model = SRNOInspired().to(device)

# 2️ Load best model weights ONLY
model.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints4/80k_30e/best_model.pth",
    map_location=device
))

model.eval()

input_folder = "C:/Users/Mardyson Justin/Thesis/VisDrone_SR/Val_HR/"
output_folder = "C:/Users/Mardyson Justin/Thesis/VisDrone_SR/sr_outputs_val/"
os.makedirs(output_folder, exist_ok=True)

scale = 3.2
image_paths = sorted(glob.glob(input_folder + "*.png"))

print("Number of images found:", len(image_paths))

with torch.no_grad():
    torch.cuda.empty_cache()
    for i, img_path in enumerate(image_paths[:50]):

        hr = Image.open(img_path).convert("RGB")

        # 1️ Create synthetic LR
        lr_img = hr.resize(
            (int(hr.width / scale), int(hr.height / scale)),
            Image.BICUBIC
        )

        # 2️ Convert to tensor
        lr = TF.to_tensor(lr_img).unsqueeze(0).to(device)

        # 3️ Reconstruct to original size
        H, W = hr.height, hr.width
        sr = model(lr, H, W)

        sr_img = TF.to_pil_image(sr.squeeze(0).clamp(0, 1))

        filename = os.path.basename(img_path)
        sr_img.save(os.path.join(output_folder, filename))

        print(f"Processed {i+1}")


Number of images found: 40169
Processed 1
Processed 2
Processed 3
Processed 4
Processed 5
Processed 6
Processed 7
Processed 8
Processed 9
Processed 10
Processed 11
Processed 12
Processed 13
Processed 14
Processed 15
Processed 16
Processed 17
Processed 18
Processed 19
Processed 20
Processed 21
Processed 22
Processed 23
Processed 24
Processed 25
Processed 26
Processed 27
Processed 28
Processed 29
Processed 30
Processed 31
Processed 32
Processed 33
Processed 34
Processed 35
Processed 36
Processed 37
Processed 38
Processed 39
Processed 40
Processed 41
Processed 42
Processed 43
Processed 44
Processed 45
Processed 46
Processed 47
Processed 48
Processed 49
Processed 50


OUTPUT Selected image

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("/content/sr_output.png") # Edit mo nalang path
plt.imshow(img)
plt.axis("off")

Validation

In [1]:
pip install lpips

  Obtaining dependency information for lpips from https://files.pythonhosted.org/packages/9b/13/1df50c7925d9d2746702719f40e864f51ed66f307b20ad32392f1ad2bb87/lpips-0.1.4-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/53.8 kB ? eta -:--:--
   --------------- ------------------------ 20.5/53.8 kB 330.3 kB/s eta 0:00:01
   ---------------------------------------- 53.8/53.8 kB 558.0 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install torchmetrics

  Obtaining dependency information for torchmetrics from https://files.pythonhosted.org/packages/02/21/aa0f434434c48490f91b65962b1ce863fdcce63febc166ca9fe9d706c2b6/torchmetrics-1.8.2-py3-none-any.whl.metadata
  Obtaining dependency information for lightning-utilities>=0.8.0 from https://files.pythonhosted.org/packages/25/f4/ead6e0e37209b07c9baa3e984ccdb0348ca370b77cea3aaea8ddbb097e00/lightning_utilities-0.15.3-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/983.2 kB ? eta -:--:--
   - ------------------------------------- 41.0/983.2 kB 991.0 kB/s eta 0:00:01
   ---- ----------------------------------- 112.6/983.2 kB 1.3 MB/s eta 0:00:01
   ----------- ---------------------------- 276.5/983.2 kB 2.1 MB/s eta 0:00:01
   ------------------------ --------------- 614.4/983.2 kB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 983.2/983.2 kB 4.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Data Cropping For Validation

In [3]:
import os
from PIL import Image

img_dir = "C:/Users/Mardyson Justin/Thesis/VisDrone2019-DET-val/images"
ann_dir = "C:/Users/Mardyson Justin/Thesis/VisDrone2019-DET-val/annotations"
out_dir = "C:/Users/Mardyson Justin/Thesis/VisDrone_SR/Val_HR"

os.makedirs(out_dir, exist_ok=True)

count = 0

for file in os.listdir(img_dir):
    if not file.endswith(".jpg"):
        continue

    img = Image.open(os.path.join(img_dir, file)).convert("RGB")
    W, H = img.size

    ann_path = os.path.join(ann_dir, file.replace(".jpg", ".txt"))
    if not os.path.exists(ann_path):
        continue

    with open(ann_path) as f:
        lines = f.readlines()

    for line in lines:
        x, y, w, h, *_ = map(int, line.split(","))

        pad = max(8, min(w, h) // 2)

        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(W, x + w + pad)
        y2 = min(H, y + h + pad)

        crop = img.crop((x1, y1, x2, y2))
        crop.save(f"{out_dir}/{count:06d}.png")
        count += 1

print("✅ Validation HR ROIs created:", count)

✅ Validation HR ROIs created: 40169


Actual Validation

In [5]:
pip install thop

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import torch
from PIL import Image
import torchvision.transforms.functional as TF
import os
import glob
import torch.nn.functional as F
from torchmetrics.functional import structural_similarity_index_measure as ssim
import lpips
import time
from thop import profile

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1️⃣ Recreate model
model = SRNOInspired().to(device)

# 2️⃣ Load best model weights
model.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints4/180k_30e/best_model.pth",
    map_location=device
))
model.eval()

# Folders
input_folder = "C:/Users/Mardyson Justin/Thesis/VisDrone_SR/Val_HR/"
output_folder = "C:/Users/Mardyson Justin/Thesis/VisDrone_SR/sr_outputs_val2/"
os.makedirs(output_folder, exist_ok=True)

# Scale
scale = 3.2
image_paths = sorted(glob.glob(input_folder + "*.png"))[:50]

# LPIPS model
lpips_fn = lpips.LPIPS(net='alex').to(device)

# Metrics accumulators
psnr_list, ssim_list, lpips_list = [], [], []
time_list = []

# PSNR calculation
def calc_psnr(sr, hr):
    mse = F.mse_loss(sr, hr)
    return 10 * torch.log10(1 / mse)

# FLOPs calculation (for first image only, optional)
flops, params = None, None

with torch.no_grad():
    for i, img_path in enumerate(image_paths):
        # Load HR image
        img = Image.open(img_path).convert("RGB")
        lr = TF.to_tensor(img).unsqueeze(0).to(device)

        H, W = img.size[1], img.size[0]
        out_h, out_w = int(H * scale), int(W * scale)

        # Measure inference time
        start_time = time.time()
        sr = model(lr, out_h, out_w)
        elapsed = time.time() - start_time
        time_list.append(elapsed)

        sr_img = TF.to_pil_image(sr.squeeze(0).clamp(0,1))

        # Resize HR to match SR
        hr_resized = TF.resize(img, (out_h, out_w))
        hr_tensor = TF.to_tensor(hr_resized).unsqueeze(0).to(device)
        sr_tensor = sr.clamp(0,1)

        # Metrics
        psnr_val = calc_psnr(sr_tensor, hr_tensor).item()
        ssim_val = ssim(sr_tensor, hr_tensor).item()
        sr_lpips = sr_tensor * 2 - 1
        hr_lpips = hr_tensor * 2 - 1
        lpips_val = lpips_fn(sr_lpips, hr_lpips).item()

        psnr_list.append(psnr_val)
        ssim_list.append(ssim_val)
        lpips_list.append(lpips_val)

        # Save SR image
        sr_img.save(os.path.join(output_folder, os.path.basename(img_path)))

        # Compute FLOPs for first image only (slow)
        if i == 0:
            flops, params = profile(model, inputs=(lr, out_h, out_w))

        print(f"[{i+1}/{len(image_paths)}] PSNR: {psnr_val:.2f}, SSIM: {ssim_val:.4f}, LPIPS: {lpips_val:.4f}, Time: {elapsed:.3f}s")

# Average metrics
print("\n✅ Average Metrics:")
print(f"PSNR: {sum(psnr_list)/len(psnr_list):.2f}")
print(f"SSIM: {sum(ssim_list)/len(ssim_list):.4f}")
print(f"LPIPS: {sum(lpips_list)/len(lpips_list):.4f}")
print(f"Average Inference Time per Image: {sum(time_list)/len(time_list):.3f}s")

if flops and params:
    print(f"FLOPs (first image): {flops/1e9:.2f} GFLOPs")
    print(f"Parameters: {params/1e6:.2f} M")


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\lpips\weights\v0.1\alex.pth


c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\torchmetrics\utilities\prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[1/50] PSNR: 28.97, SSIM: 0.8998, LPIPS: 0.1635, Time: 0.315s
[2/50] PSNR: 27.20, SSIM: 0.8895, LPIPS: 0.1941, Time: 0.008s
[3/50] PSNR: 26.34, SSIM: 0.8597, LPIPS: 0.1905, Time: 0.026s
[4/50] PSNR: 30.87, SSIM: 0.9068, LPIPS: 0.1676, Time: 0.009s
[5/50] PSNR: 28.45, SSIM: 0.8641, LPIPS: 0.1751, Time: 0.004s
[6/50] PSNR: 28.77, SSIM: 0.8813, LPIPS: 0.1842, Time: 0.003s
[7/50] PSNR: 28.42, SSIM: 0.8736, LPIPS: 0.2082, Time: 0.003s
[8/50] PSNR: 26.95, SSIM: 0.8502, LPIPS: 0.1823, Time: 0.003s
[9/50] PSNR: 26.93, SSIM: 0.8483, LPIPS: 0.1850, Time: 0.004s
[10/50] PSNR: 28.31, SSIM: 0.8749, LPIPS: 0.1519, Time: 0.002s
[11/50] PSNR: 27.99, SSIM: 0.8616, LPIPS: 0.1849, Time: 0.004s
[12/50] P

In [8]:
import torch
from PIL import Image
import torchvision.transforms.functional as TF
import os
import glob
import torch.nn.functional as F
from torchmetrics.functional import structural_similarity_index_measure as ssim
import lpips
import time
from thop import profile
import pandas as pd

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1️⃣ Recreate model
model = SRNOInspired().to(device)

# 2️⃣ Load best model weights
model.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints4/220k_30e/best_model.pth",
    map_location=device
))
model.eval()

# Folders
input_folder = "C:/Users/Mardyson Justin/Thesis/VisDrone_SR/Val_HR/"
output_folder = "C:/Users/Mardyson Justin/Thesis/VisDrone_SR/sr_outputs_val2/"
os.makedirs(output_folder, exist_ok=True)

# Results folder
results_folder = "C:/Users/Mardyson Justin/Thesis/SRNO_Validation_Results/"
os.makedirs(results_folder, exist_ok=True)
excel_path = os.path.join(results_folder, "220k_metrics.xlsx")

# Scale factor
scale = 3.2

# Collect images (.png + .jpg)
image_paths = sorted(
    glob.glob(os.path.join(input_folder, "*.png")) +
    glob.glob(os.path.join(input_folder, "*.jpg"))
)[:50]

print("Images found:", len(image_paths))

# LPIPS model
lpips_fn = lpips.LPIPS(net='alex').to(device)

# Storage
results_data = []
psnr_list, ssim_list, lpips_list, time_list = [], [], [], []

# PSNR function
def calc_psnr(sr, hr):
    mse = F.mse_loss(sr, hr)
    return 10 * torch.log10(1 / mse)

flops, params = None, None

with torch.no_grad():
    for i, img_path in enumerate(image_paths):

        # Load HR image
        hr_img = Image.open(img_path).convert("RGB")

        # Create synthetic LR (degradation)
        lr_img = hr_img.resize(
            (int(hr_img.width / scale), int(hr_img.height / scale)),
            Image.BICUBIC
        )

        lr = TF.to_tensor(lr_img).unsqueeze(0).to(device)

        H, W = hr_img.height, hr_img.width

        # Inference time
        start_time = time.time()
        sr = model(lr, H, W)
        elapsed = time.time() - start_time
        time_list.append(elapsed)

        sr = sr.clamp(0, 1)
        sr_img = TF.to_pil_image(sr.squeeze(0))

        # Resize HR to match SR size (for fair comparison)
        hr_resized = TF.to_tensor(hr_img).unsqueeze(0).to(device)

        # Metrics
        psnr_val = calc_psnr(sr, hr_resized).item()
        ssim_val = ssim(sr, hr_resized).item()

        sr_lpips = sr * 2 - 1
        hr_lpips = hr_resized * 2 - 1
        lpips_val = lpips_fn(sr_lpips, hr_lpips).item()

        psnr_list.append(psnr_val)
        ssim_list.append(ssim_val)
        lpips_list.append(lpips_val)

        # Save SR image
        filename = os.path.basename(img_path)
        sr_img.save(os.path.join(output_folder, filename))

        # Save row data
        results_data.append({
            "Image": filename,
            "PSNR": psnr_val,
            "SSIM": ssim_val,
            "LPIPS": lpips_val,
            "Inference_Time_sec": elapsed
        })

        # FLOPs (first image only)
        if i == 0:
            flops, params = profile(model, inputs=(lr, H, W), verbose=False)

        print(f"[{i+1}/{len(image_paths)}] "
              f"PSNR: {psnr_val:.2f}, "
              f"SSIM: {ssim_val:.4f}, "
              f"LPIPS: {lpips_val:.4f}, "
              f"Time: {elapsed:.4f}s")

# Convert to DataFrame
df = pd.DataFrame(results_data)

# Compute averages
avg_psnr = df["PSNR"].mean()
avg_ssim = df["SSIM"].mean()
avg_lpips = df["LPIPS"].mean()
avg_time = df["Inference_Time_sec"].mean()

# Add average row
avg_row = pd.DataFrame([{
    "Image": "AVERAGE",
    "PSNR": avg_psnr,
    "SSIM": avg_ssim,
    "LPIPS": avg_lpips,
    "Inference_Time_sec": avg_time
}])

df = pd.concat([df, avg_row], ignore_index=True)

# Save Excel
df.to_excel(excel_path, index=False)

print("\n✅ Average Metrics:")
print(f"PSNR: {avg_psnr:.2f}")
print(f"SSIM: {avg_ssim:.4f}")
print(f"LPIPS: {avg_lpips:.4f}")
print(f"Average Inference Time per Image: {avg_time:.4f}s")

if flops and params:
    print(f"FLOPs (first image): {flops/1e9:.2f} GFLOPs")
    print(f"Parameters: {params/1e6:.2f} M")

print(f"\n📊 Excel file saved at:\n{excel_path}")

Images found: 50
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\lpips\weights\v0.1\alex.pth
[1/50] PSNR: 26.15, SSIM: 0.8319, LPIPS: 0.2147, Time: 0.0040s
[2/50] PSNR: 24.99, SSIM: 0.8394, LPIPS: 0.1504, Time: 0.0040s
[3/50] PSNR: 23.01, SSIM: 0.7537, LPIPS: 0.3042, Time: 0.0043s


c:\Users\Mardyson Justin\Thesis\venv\Lib\site-packages\torchmetrics\utilities\prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


[4/50] PSNR: 28.40, SSIM: 0.7950, LPIPS: 0.1488, Time: 0.0022s
[5/50] PSNR: 26.11, SSIM: 0.7712, LPIPS: 0.1025, Time: 0.0042s
[6/50] PSNR: 26.08, SSIM: 0.7714, LPIPS: 0.1529, Time: 0.0031s
[7/50] PSNR: 24.98, SSIM: 0.7948, LPIPS: 0.0992, Time: 0.0039s
[8/50] PSNR: 24.07, SSIM: 0.7377, LPIPS: 0.1401, Time: 0.0030s
[9/50] PSNR: 23.69, SSIM: 0.7389, LPIPS: 0.2137, Time: 0.0030s
[10/50] PSNR: 24.76, SSIM: 0.7950, LPIPS: 0.0383, Time: 0.0035s
[11/50] PSNR: 24.72, SSIM: 0.7253, LPIPS: 0.2241, Time: 0.0040s
[12/50] PSNR: 26.30, SSIM: 0.7599, LPIPS: 0.1344, Time: 0.0040s
[13/50] PSNR: 21.78, SSIM: 0.6539, LPIPS: 0.2837, Time: 0.0041s
[14/50] PSNR: 21.80, SSIM: 0.5977, LPIPS: 0.3266, Time: 0.0050s
[15/50] PSNR: 21.54, SSIM: 0.6084, LPIPS: 0.3465, Time: 0.0032s
[16/50] PSNR: 21.68, SSIM: 0.5961, LPIPS: 0.3762, Time: 0.0030s
[17/50] PSNR: 21.04, SSIM: 0.6167, LPIPS: 0.2994, Time: 0.0044s
[18/50] PSNR: 21.09, SSIM: 0.6279, LPIPS: 0.4324, Time: 0.0034s
[19/50] PSNR: 21.89, SSIM: 0.5748, LPIPS: 0.49

## LOAD Dataset

In [6]:
import torch
import numpy as np
import cv2
import os
from ultralytics import YOLO
from PIL import Image
import torchvision.transforms.functional as TF

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load detector
detector = YOLO("C:/Users/Mardyson Justin/Thesis/yolov9c_visdrone_finetune15/weights/best.pt")

# Load SRNO
sr_model = SRNOInspired().to(device)
sr_model.load_state_dict(torch.load(
    "C:/Users/Mardyson Justin/Thesis/SRNO_Checkpoints4/180k_30e/best_model.pth",
    map_location=device
))
sr_model.eval()

SRNOInspired(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (mlp): Sequential(
    (0): Linear(in_features=258, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=3, bias=True)
  )
)